# Homework: The EM Algorithm

## Problem 1: EM for a Two-Component Exponential Mixture

Suppose you observe $n$ data points $y_1, \ldots, y_n$ from a two-component exponential mixture:

$$f(y_i | \boldsymbol{\theta}) = \pi \, \lambda_1 e^{-\lambda_1 y_i} + (1 - \pi) \, \lambda_2 e^{-\lambda_2 y_i}, \quad y_i > 0$$

where $\boldsymbol{\theta} = (\pi, \lambda_1, \lambda_2)$.

**(a)** Write down the complete-data log-likelihood, treating the component membership $z_i \in \{1, 2\}$ as the latent variable.

$\prod_{i=1}^{n} (\pi \lambda_1 e^{-\lambda_1 y_i})^{I(z_i = 1)} ((1 - \pi) \lambda_2 e^{-\lambda_2 y_i})^{I(z_i = 2)}$

**(b)** Derive the E-step: compute the responsibility $\gamma_i^{(t)} = P(z_i = 1 | y_i, \boldsymbol{\theta}^{(t)})$.

$P(z_i = 1 | y_i, \boldsymbol{\theta}^{(t)}) = \frac{P(z_i = 1 | \boldsymbol{\theta}^{(t)})f(y_i | z_i = 1, \boldsymbol{\theta}^{(t)})}{f(y_i | \boldsymbol{\theta}^{(t)})}
= \frac{\pi^{(t)} \lambda_1^{(t)} e^{-\lambda_1^{(t)} y_i}}{\pi^{(t)} \, \lambda_1^{(t)} e^{-\lambda_1^{(t)} y_i} + (1 - \pi^{(t)}) \, \lambda_2^{(t)} e^{-\lambda_2^{(t)} y_i}}$


**(c)** Derive the M-step: write the update formulas for $\pi^{(t+1)}$, $\lambda_1^{(t+1)}$, and $\lambda_2^{(t+1)}$.

$Q(\boldsymbol{\theta} | \boldsymbol{\theta}^{(t)}) = \sum_{i=1}^n (\gamma_i^{(t)}(log\pi + log\lambda_1 - \lambda_1y_i) + (1 - \gamma_i^{(t)})(log(1 -\pi) + log\lambda_2 - \lambda_2 y_i))$

$\frac{\partial Q}{\partial \pi} = \sum_{i=1}^n (\gamma_i^{(t)}\frac{1}{\pi} - (1 - \gamma_i^{(t)})\frac{1}{1 - \pi}) = 0$

$\implies \pi^{(t + 1)} = \frac{\sum_{i=1}^n \gamma_i^{(t)}}{n}$

$\frac{\partial Q}{\partial \lambda_1} = \sum_{i=1}^n \gamma_i^{(t)} (\frac{1}{\lambda_1} - y_i) = 0$

$\implies \lambda_1^{(t + 1)} = \frac{\sum_{i=1}^n \gamma_i^{(t)}}{\sum_{i=1}^n \gamma_i^{(t)} y_i}$

$\frac{\partial Q}{\partial \lambda_2} = \sum_{i=1}^n (1 - \gamma_i^{(t)}) (\frac{1}{\lambda_2} - y_i) = 0$

$\implies \lambda_2^{(t + 1)} = \frac{\sum_{i=1}^n (1 - \gamma_i^{(t)})}{\sum_{i=1}^n (1 - \gamma_i^{(t)}) y_i}$


**(d)** Implement the EM algorithm for this model. Your function should have the following signature and return a dictionary containing the estimated parameters and the log-likelihood history.

In [ ]:
import numpy as np
from scipy import stats

def em_exponential_mixture(y, pi_init, lam1_init, lam2_init,
                           max_iter=200, tol=1e-8):
    """EM algorithm for a 2-component exponential mixture.

    Parameters
    ----------
    y : np.ndarray
        Observed data of shape (n,), all positive.
    pi_init : float
        Initial mixing proportion for component 1.
    lam1_init, lam2_init : float
        Initial rate parameters.
    max_iter : int
        Maximum number of iterations.
    tol : float
        Convergence tolerance on log-likelihood change.

    Returns
    -------
    dict with keys: pi, lam1, lam2, loglik_history
    """
    
    n = len(y)
    pi = pi_init
    lam1 = lam1_init
    lam2 = lam2_init

    # history = []
    loglik_history = []

    for iter in range(max_iter):
        pi_old = pi
        lam1_old = lam1
        lam2_old = lam2

        # E-step
        d1 = compute_d1(pi, lam1, y)
        d2 = compute_d2(pi, lam2, y)
        gamma = d1 / (d1 + d2)

        # M-step
        pi = np.sum(gamma) / n
        lam1 = np.sum(gamma) / (gamma @ y)
        lam2 = np.sum(1 - gamma) / ((1 - gamma) @ y)

        # observed data log likelihood
        if iter == 0:
            loglik = np.sum(np.log(d1 + d2))
        
        loglik_old = loglik

        d1_new = compute_d1(pi, lam1, y)
        d2_new = compute_d2(pi, lam2, y)
        loglik = np.sum(np.log(d1_new + d2_new))
        
        # history.append({
        #     "iter": iter + 1, "pi": pi, "lam1": lam1, "lam2": lam2, "loglik": loglik
        # })
        loglik_history.append(loglik)

        loglik_change = abs(loglik - loglik_old)

        if loglik_change < tol:
            break

    if iter == max_iter - 1 and  loglik_change >= tol:
        print("Warning: Iteration limit reached without convergence!")
    
    return {"pi": pi, "lam1": lam1, "lam2": lam2, "loglik_history": loglik_history}

def compute_d1(pi, lam1, y):
    return pi * lam1 * np.exp(- lam1 * y)

def compute_d2(pi, lam2, y):
    return (1 - pi) * lam2 * np.exp(- lam2 * y)
    

Test your implementation on the following simulated data and report the estimated parameters:

In [40]:
np.random.seed(123)
n = 500
pi_true = 0.35
lam1_true, lam2_true = 0.5, 3.0
z = np.random.binomial(1, 1 - pi_true, n)
y = np.where(z == 0,
             np.random.exponential(1 / lam1_true, n),
             np.random.exponential(1 / lam2_true, n))

result = em_exponential_mixture(y, pi_init=0.5, lam1_init=0.3, lam2_init=2.0)

In [ ]:
print("EM estimates vs. true values:")
print(f"  pi:     {result['pi']:.3f}  (true: {pi_true})")
print(f"  lam1:    {result['lam1']:.2f}  (true: {lam1_true})")
print(f"  lam2:    {result['lam2']:.2f}  (true: {lam2_true})")
print(f"  Iterations: {len(result['loglik_history'])}")

EM estimates vs. true values:
  pi:     0.299  (true: 0.35)
  lam1:    0.44  (true: 0.5)
  lam2: 2.48  (true: 3.0)
  Iterations: 93


## Problem 2: Diagnosing EM Convergence

A colleague provides the following EM implementation for the two-component Gaussian mixture model. Read the code carefully and answer the questions below.

In [ ]:
import numpy as np
from scipy import stats

def em_gmm_v2(y, pi_init, mu1_init, sigma1_init, mu2_init, sigma2_init,
              max_iter=200, tol=1e-8):
    n = len(y)
    pi = pi_init
    mu1, mu2 = mu1_init, mu2_init
    s1, s2 = sigma1_init, sigma2_init
    loglik_history = []

    for iteration in range(max_iter):
        d1 = pi * stats.norm.pdf(y, mu1, s1)
        d2 = (1 - pi) * stats.norm.pdf(y, mu2, s2)
        gamma = d1 / (d1 + d2)

        ll = np.sum(np.log(d1 + d2))
        loglik_history.append(ll)

        if iteration > 0 and loglik_history[-1] - loglik_history[-2] < tol:
            break

        n1 = np.sum(gamma)
        n2 = n - n1

        pi = n1 / n
        mu1 = np.sum(gamma * y) / n1
        mu2 = np.sum((1 - gamma) * y) / n2
        s1 = np.sqrt(np.sum(gamma * (y - mu1) ** 2) / n1)
        s2 = np.sqrt(np.sum((1 - gamma) * (y - mu2) ** 2) / n2)

    return {
        "pi": pi, "mu1": mu1, "sigma1": s1,
        "mu2": mu2, "sigma2": s2,
        "loglik_history": loglik_history,
    }

**(a)** There is a subtle bug in the convergence check. Identify it and explain what incorrect behavior it could cause.


We should have written `abs(loglik_history[-1] - loglik_history[-2]) < tol` instead.

If the log-likelihood decreases somehow, then the loop will stop immediately.


**(b)** Suppose this function is called with `sigma1_init=0.001` on a dataset where one observation happens to be very close to `mu1_init`. Explain what numerical issue could arise during the E-step and how you would fix it.


`d1` will be very large for this observation, so `gamma` will be close to 1 for this observation. Since `sigma1_init` is very small, then `gamma` will be close to 0 for other observations.
As a result, this observation very close to `mu1_init`  dominates the update of the first componenet, causing `mu1` to move onto that observation and `sigma1` to move toward 0.

I will use a larger value for `sigma1_init`. We can also use K-means clustering to get initial componenent assignments and then compute initial parameter estimates from these clusters.


**(c)** Does this implementation handle the label-switching symmetry of the mixture model? That is, if you swap the roles of component 1 and component 2 in the initialization, do you get an equivalent solution? Explain why or why not.

Yes. If we swap the roles of component 1 and component 2 in the initialization, then the new `pi_init` is essentially the old `1 - pi_init`; `mu1_init, sigma1_init` and `mu2_init, sigma2_init` are exchanged.
At E-step, the new `gamma` becomes the old `1 - gamma`. At M-step, the updated `pi` becomes `1 - pi`. The updated `mu1, sigma1` and `mu2, sigma2` are exchanged. So, everything is just relabeled.


## Problem 3: EM for Missing Data

Consider a dataset of $n = 200$ paired measurements $(X_i, Y_i)$ from a bivariate normal distribution, where 25% of $Y$ values are missing at random.

**(a)** Write a function that implements the EM algorithm for estimating the bivariate normal parameters $(\mu_X, \mu_Y, \sigma_X^2, \sigma_Y^2, \rho)$ when $Y$ has missing values. Use the following signature:

In [78]:
def em_bivariate(X, Y, observed, max_iter=200, tol=1e-8):
    """EM for bivariate normal with missing Y values.

    Parameters
    ----------
    X : np.ndarray
        Fully observed variable, shape (n,).
    Y : np.ndarray
        Partially observed variable, shape (n,). NaN for missing.
    observed : np.ndarray
        Boolean array, True where Y is observed.
    max_iter : int
        Maximum iterations.
    tol : float
        Convergence tolerance on max parameter change.

    Returns
    -------
    dict with keys: mu_x, mu_y, var_x, var_y, rho, n_iter
    """
    
    n = len(X)
    mis_idx = np.where(~observed)[0]
    y = Y.copy() # create a copy
    
    y_square = y**2
    xy = X * y

    # initialization
    mu_x = np.mean(X)
    mu_y = np.nanmean(y)
    var_x = np.var(X, ddof = 0) # compute MLE by setting ddof = 0 (which is default)
    var_y = np.nanvar(y, ddof = 0)
    rho = np.corrcoef(X[observed], y[observed])[0, 1]

    for iter in range(max_iter):
        # mu_x_old = mu_x
        mu_y_old = mu_y
        # var_x_old = var_x
        var_y_old = var_y
        rho_old = rho

        #####################################################
        # E-step

        # E[y_mis | obs]
        y[mis_idx] = mu_y + rho * np.sqrt(var_y) / np.sqrt(var_x) * (X - mu_x)[mis_idx]

        # E[y_mis^2 | obs]
        y_square[mis_idx] = (1 - rho**2) * var_y + y[mis_idx]**2

        # E[x*y_mis | obs]
        xy[mis_idx] = (X * y)[mis_idx]

        #####################################################
        # M-step
        mu_y = np.mean(y)
        var_y = 1/n * np.sum(y_square) - mu_y**2
        rho = (1/n * np.sum(xy) - mu_x * mu_y)/np.sqrt(var_x)/np.sqrt(var_y)

        # no need to update mu_x, var_x at M-step

        param_change = np.max([abs(mu_y - mu_y_old), abs(var_y - var_y_old), abs(rho - rho_old)])
        if param_change < tol:
            break

    if iter == max_iter - 1 and  param_change >= tol:
        print("Warning: Iteration limit reached without convergence!")    

    return {
        "mu_x": mu_x, "mu_y": mu_y, "var_x": var_x, "var_y": var_y, 
        "rho": rho, "n_iter": iter + 1
    }


**(b)** Test your implementation on the following simulated data. Compare the EM estimates with the complete-case estimates (using only observations where $Y$ is observed) and the full-data estimates (using the true $Y$ values before deletion).

In [80]:
np.random.seed(42)
n = 200
mu = np.array([3.0, 7.0])
rho_true = 0.6
sx, sy = 1.5, 2.0
Sigma = np.array([[sx**2, rho_true * sx * sy],
                   [rho_true * sx * sy, sy**2]])

data = np.random.multivariate_normal(mu, Sigma, n)
X, Y_full = data[:, 0], data[:, 1]

# Introduce 25% MAR missingness
prob_miss = 0.25 * np.ones(n)
missing = np.random.binomial(1, prob_miss, n).astype(bool)
Y = Y_full.copy()
Y[missing] = np.nan
observed = ~missing

In [81]:
result = em_bivariate(X, Y, observed)
result

{'mu_x': np.float64(2.9575587837375408),
 'mu_y': np.float64(7.021104128288585),
 'var_x': np.float64(2.008791570670503),
 'var_y': np.float64(3.826728251137446),
 'rho': np.float64(0.5826090468613828),
 'n_iter': 11}

In [95]:
print("EM estimates vs. Complete-case estimates vs. True estimates if no deletion:")
print(f"  mu_x:    EM: {result['mu_x']:.3f}  Complete-case: {np.mean(X[observed]):.3f} True estimate: {np.mean(X):.3f} True value: {mu[0]}")
print(f"  mu_y:    EM: {result['mu_y']:.3f}  Complete-case: {np.mean(Y[observed]):.3f} True estimate: {np.mean(Y_full):.3f} True value: {mu[1]}")
print(f"  var_x:   EM: {result['var_x']:.3f} Complete-case: {np.var(X[observed]):.3f}  True estimate: {np.var(X):.3f} True value: {sx**2}")
print(f"  var_y:  EM: {result['var_y']:.3f} Complete-case: {np.var(Y[observed]):.3f}   True estimate: {np.var(Y_full):.3f} True value: {sy**2}" ) 
print(f"  rho:    EM: {result['rho']:.3f}  Complete-case: {np.corrcoef(X[observed], Y[observed])[0,1]:.3f}  True estimate: {np.corrcoef(X, Y_full)[0, 1]:.3f} True value: {rho_true}" ) 
print(f"  Iterations: {result['n_iter']}")

EM estimates vs. Complete-case estimates vs. True estimates if no deletion:
  mu_x:    EM: 2.958  Complete-case: 2.916 True estimate: 2.958 True value: 3.0
  mu_y:    EM: 7.021  Complete-case: 6.988 True estimate: 7.010 True value: 7.0
  var_x:   EM: 2.009 Complete-case: 2.030  True estimate: 2.009 True value: 2.25
  var_y:  EM: 3.827 Complete-case: 3.841   True estimate: 3.712 True value: 4.0
  rho:    EM: 0.583  Complete-case: 0.585  True estimate: 0.589 True value: 0.6
  Iterations: 11


These three types of estimates are very close. In fact, the missingness mechanism is not MAR but MCAR.

**(c)** Which estimator do you expect to have smaller variance for $\rho$, the EM estimator or the complete-case estimator? Explain why.

The EM estimator. Complete-case estimator drops some data points with missing $Y$ s, so it loses information. Thus it is less efficient.


## Problem 4: Zero-Inflated Poisson Model Selection

The following code fits a standard Poisson model and a zero-inflated Poisson (ZIP) model to a dataset. Read the code and answer the questions.

In [ ]:
import numpy as np
from scipy import stats

def fit_poisson(y):
    """Fit a standard Poisson model by MLE."""
    lam_hat = np.mean(y)
    ll = np.sum(stats.poisson.logpmf(y, lam_hat))
    return {"lam": lam_hat, "loglik": ll, "n_params": 1}

def fit_zip_em(y, max_iter=200, tol=1e-8):
    """Fit a zero-inflated Poisson model by EM."""
    n = len(y)
    pi = 0.3
    lam = np.mean(y[y > 0])
    is_zero = (y == 0)
    loglik_history = []

    for iteration in range(max_iter):
        gamma = np.zeros(n)
        gamma[is_zero] = pi / (pi + (1 - pi) * np.exp(-lam))

        ll = (np.sum(is_zero) * np.log(pi + (1 - pi) * np.exp(-lam))
              + np.sum(~is_zero * (np.log(1 - pi)
                                    + stats.poisson.logpmf(y, lam))))
        loglik_history.append(ll)

        if iteration > 0 and abs(loglik_history[-1] - loglik_history[-2]) < tol:
            break

        pi = np.mean(gamma)
        lam = np.sum((1 - gamma) * y) / np.sum(1 - gamma)

    return {"pi": pi, "lam": lam, "loglik": ll,
            "n_params": 2, "loglik_history": loglik_history}

**(a)** In the E-step, why is $\gamma_i = 0$ for all observations with $y_i > 0$? Explain using the structure of the ZIP model.

**(b)** Given that the Poisson model is nested within the ZIP model (setting $\pi = 0$ recovers the Poisson), a natural model comparison tool is the likelihood ratio test. However, the standard chi-squared approximation for the LRT is not valid here. Explain why, in terms of the parameter space and the null hypothesis.

**(c)** Using the AIC ($\text{AIC} = -2\ell + 2k$, where $k$ is the number of parameters), compare the Poisson and ZIP fits on the following data. Which model does AIC prefer?

In [ ]:
np.random.seed(10)
n = 300
pi_true = 0.2
lam_true = 2.0
z = np.random.binomial(1, pi_true, n)
y = np.where(z == 1, 0, np.random.poisson(lam_true, n))

poisson_fit = fit_poisson(y)
zip_fit = fit_zip_em(y)

## Problem 5: EM with Asymmetric Components

Consider a two-component mixture where component 1 is $\text{Exponential}(\lambda)$ (supported on $[0, \infty)$) and component 2 is $N(\mu, \sigma^2)$ (supported on $(-\infty, \infty)$):

$$f(y_i | \boldsymbol{\theta}) = \pi \, \lambda e^{-\lambda y_i} \mathbb{1}(y_i > 0) + (1 - \pi) \, \phi(y_i | \mu, \sigma^2)$$

**(a)** Derive the E-step and M-step for this model. Pay careful attention to how the exponential component's support restriction affects the responsibility computation for observations with $y_i \leq 0$.

**(b)** Implement the EM algorithm for this model:

In [ ]:
def em_exp_normal_mixture(y, pi_init, lam_init, mu_init, sigma_init,
                          max_iter=200, tol=1e-8):
    """EM for exponential-normal mixture.

    Parameters
    ----------
    y : np.ndarray
        Observed data of shape (n,).
    pi_init : float
        Initial mixing proportion for the exponential component.
    lam_init : float
        Initial rate for the exponential component.
    mu_init : float
        Initial mean for the normal component.
    sigma_init : float
        Initial std dev for the normal component.
    max_iter : int
        Maximum iterations.
    tol : float
        Convergence tolerance on log-likelihood change.

    Returns
    -------
    dict with keys: pi, lam, mu, sigma, loglik_history
    """
    # Your code here
    pass

**(c)** Test on the following data and report your estimates:

In [ ]:
np.random.seed(42)
n = 400
pi_true = 0.4
lam_true = 2.0
mu_true, sigma_true = -1.0, 1.5
z = np.random.binomial(1, pi_true, n)
y = np.where(z == 1,
             np.random.exponential(1 / lam_true, n),
             np.random.normal(mu_true, sigma_true, n))

result = em_exp_normal_mixture(y, pi_init=0.5, lam_init=1.0,
                                mu_init=0.0, sigma_init=2.0)